# Encoder-Based Distance Movement Script

## Overview
This script enables a robot to move **forward or backward by a specific target distance** using **wheel encoders** for precise measurement.  
It continuously monitors encoder counts, converts them into distance traveled, and stops the robot once the target distance is reached.

**Full Coding available at [Motor_and_Encoder.py](Motor_and_Encoder.py)** 

## Features
- **Encoder Integration**: Uses wheel encoders for accurate distance tracking.  
- **Directional Control**: Supports both forward and backward movement.  
- **Live Feedback**: Prints encoder counts and distance in meters during motion.  
- **Safe Stop**: Automatically brakes the robot when the target distance is reached.  
- **Graceful Cleanup**: Ensures proper resource release on exit.  

## Libraries Used
- **RPi_Robot_Hat_Lib** → Custom motor, servo, and encoder controller  
- **time** → Timing delays for loop control  




## Let's Start Coding ! 
### 1. Import Required Libraries 
Import the necessary libraries for motor control and timing:

- **RPi_Robot_Hat_Lib**: Custom library providing robot control and encoder functionality
- **time**: Built-in Python library for timing operations and delays

In [ ]:
from RPi_Robot_Hat_Lib import RobotController 
import time

### 1. The Function `init()`

This function initializes the **motor controller** and **encoder system**.

- **Function Name**: `init`

- **Global Variables**:  
  - `Motor`: The robot motor controller  
  - `enc`: Encoder handler (using the same `Motor` object)  

- **Usage**:  
  - Creates a new `RobotController` instance with wheel diameter set to **98 mm**  
  - Assigns `enc = Motor` for encoder distance reading  
    ```python
    Motor = RobotController(wheel_diameter=98)
    enc = Motor
    ```

In [ ]:
def init():
    global Motor, enc
    Motor = RobotController(wheel_diameter=98)  # diameter in mm
    enc = Motor  # Use same object for encoder functions

### 2. The Function `move_to_distance(target_distance, target_speed)`

This function moves the robot forward or backward until the specified distance is reached, using encoder feedback for accuracy.

- **Function Name**: `move_to_distance`

- **Parameters**:  
  - `target_distance` *(float)* → Target distance in meters (negative value means move backward).  
  - `target_speed` *(float)* → Speed percentage for the motors (range 1–100).  

- **Global Variables**:  
  - `Motor`: Motor controller instance from `RPi_Robot_Hat_Lib`.  
  - `enc`: Encoder handler object (same as `Motor`).  

- **Usage**:  
  - Moves the robot **forward** if `target_distance ≥ 0`.  
    ```python
    Motor.Forward(target_speed)
    ```
  - Moves the robot **backward** if `target_distance < 0`.  
    ```python
    Motor.Backward(target_speed)
    target_distance = abs(target_distance)
    ```
  - Continuously reads **encoder distance** and **raw counts** for both left and right wheels:  
    ```python
    left_distance = enc.get_distance('LF')
    right_distance = enc.get_distance('RF')
    left_enc = enc.get_encoder('LF')
    right_enc = enc.get_encoder('RF')
    ```
  - Prints live values:  
    ```text
    Left Encoder: 123.45
    Right Encoder: 125.67
    Left Distance: 1.23m
    Right Distance: 1.25m
    ```
  - Stops the motors once **both wheel distances** reach or exceed the `target_distance`.  
    ```python
    if right_distance >= target_distance and left_distance >= target_distance:
        Motor.Brake()
        break
    ```
  - Uses a **0.1 second loop delay** for stability.  


In [ ]:
def move_to_distance(target_distance, target_speed):
    global Motor, enc
    if target_distance >= 0:
        Motor.Forward(target_speed)  # Move forward
    else:
        Motor.Backward(target_speed)  # Move backward
        target_distance = abs(target_distance)  # Use absolute value for comparison

    while True:
        # Get the encoder counts and current distance
        left_distance = enc.get_distance('LF')
        right_distance = enc.get_distance('RF')
        
        # Get encoder values for display
        left_enc = enc.get_encoder('LF')
        right_enc = enc.get_encoder('RF')

        # Display encoder values and distances
        print("Left Encoder: {:.2f}".format(left_enc))
        print("Right Encoder: {:.2f}".format(right_enc))
        print("Left Distance: {:.2f}m".format(left_distance))
        print("Right Distance: {:.2f}m".format(right_distance))

        # Stop the motor once the target distance is reached
        if right_distance >= target_distance and left_distance >= target_distance:
            Motor.Brake()
            break

        time.sleep(0.1)

### 3. The Function `cleanup()`

This function safely shuts down the motor controller and releases GPIO resources.

- **Function Name**: `cleanup`

- **Global Variables**:  
  - `Motor`: Motor controller instance from `RPi_Robot_Hat_Lib`.  
  - `enc`: Encoder handler object (same as `Motor`).  

- **Usage**:  
  - Calls the `Motor.cleanup()` method to stop motors and release GPIO pins.  
    ```python
    Motor.cleanup()
    ```
  - Ensures a safe shutdown of the robot system.


In [ ]:
def cleanup():
    global Motor, enc
    Motor.cleanup()

### 4. The Function `main()`

This function serves as the entry point for running the distance-based motor control program.

- **Function Name**: `main`

- **Usage**:  
  - Prompts the user to input a target distance in meters (negative value for backward movement).  
    ```python
    target_distance = float(input("Enter the target distance (in meters, negative for backward): "))
    ```
  - Prompts the user to input a target speed between **1 and 100**.  
    ```python
    target_speed = float(input("Enter the target speed (1 - 100): "))
    ```
  - Calls the `move_to_distance()` function with the provided distance and speed to execute movement.  
    ```python
    move_to_distance(target_distance, target_speed)
    ```


In [ ]:
def main():
    target_distance = float(input("Enter the target distance (in meters, negative for backward): "))
    target_speed = float(input("Enter the target speed (1 - 100): "))
    move_to_distance(target_distance, target_speed)

### 5. Script Execution Block  

This block ensures the program starts correctly when executed directly and shuts down cleanly on interruption.  

- **Execution Block**:  
  - Calls the `init()` function to initialize the motor and encoder.  
    ```python
    init()
    ```
  - Runs the `main()` function to start the distance control program.  
    ```python
    main()
    ```

- **Exception Handling**:  
  - **Keyboard Interrupt** (`Ctrl + C`):  
    - Safely stops the program and calls `cleanup()` to release resources.  
    ```python
    except KeyboardInterrupt:
        print("Shutting down")
        cleanup()
    ```


In [ ]:
if __name__ == '__main__':
    try:
        init()
        main()
    except KeyboardInterrupt:
        print("Shutting down")
        cleanup()